# Hyperliquid vault of vaults - rank composite metadata selection

Based on: `70-hyperliquid-metadata-adjusted-simple-signal.ipynb`

## What changed

NB70 showed that metadata regularisation did **not** unlock better sizing. The strategy stayed stable, but `weight_equal` still won and the metadata layer mostly acted as a mild regulariser.

This notebook tests the next question directly:

- is **ordinal ranking** more reliable than **raw score magnitude**
- can `age` and `TVL` work better as **rank features** than as multiplicative credibility controls
- can we improve selection quality among gated survivors without reopening the noisy weighting problem

## Experiment: rank composite with metadata and top-k selection

The Bayesian gate stays fixed close to the NB68/NB70 winner so this notebook isolates the ranking question.

Each rebalance has three stages:

1. **Gate stage**: keep only vaults with positive Bayesian Sterling gate signal
2. **Rank stage**: rank gated survivors on a composite of performance, risk, age, and TVL features
3. **Selection and weight stage**: keep only the top-ranked vaults and compare equal weight against mild tilts

### Search dimensions

**Gate family**
- fixed `gate_signal_variant = bayesian_sterling`
- fixed `gate_mode = positive_only`
- narrow search over `volatility_window`, `bayesian_halflife`, and `sterling_constant`

**Rank feature set**
- `returns + sharpe + age + tvl`
- `returns + sharpe + drawdown + age + tvl`
- `returns + sterling + drawdown + age + tvl`

**Rank weight scheme**
- `equal_feature_weights`
- `performance_heavy`
- `credibility_heavy`

**Selection**
- `top_k`
- `top_quantile`

**Weight functions**
- `weight_equal`
- `weight_blend_0.5`
- `weight_softmax_2.0`

## Hypothesis

Relative ordering is more reliable than raw score magnitude.

If that is true, then a rank composite over gated survivors should:

- preserve the robustness of the Bayesian gate
- use `age` and `TVL` as credibility signals without over-compressing magnitudes
- make mild tilts safer than in NB70's raw-score setup

## Success condition

At least one rank-composite variant should:

- match or beat the NB68/NB70 equal-weight baselines on Sharpe or Calmar
- keep drawdown close to the current conservative baseline
- avoid materially worse concentration
- show stable top-k membership rather than noisy reshuffling every rebalance

## Research findings

_To be filled after execution._


# Set up

Set up Trading Strategy data client.


In [ ]:
import logging

from tradingstrategy.client import Client
from tradeexecutor.utils.notebook import setup_charting_and_output, OutputMode, set_notebook_logging

client = Client.create_jupyter_client()

# Set up drawing charts in interactive vector output mode.
# This is slower. See the alternative commented option below.
# setup_charting_and_output(OutputMode.interactive)

# Set up rendering static PNG images.
# This is much faster but disables zoom on any chart.
setup_charting_and_output(OutputMode.static, image_format="png", width=1500, height=1000)


logger = logging.getLogger("strategy")



# Chain configuration

- Choose target chains and their vults

In [ ]:
from eth_defi.token import USDC_NATIVE_TOKEN
from eth_defi.token import USDT_NATIVE_TOKEN
from eth_defi.token import WRAPPED_NATIVE_TOKEN

from tradingstrategy.chain import ChainId
from tradingstrategy.lending import LendingProtocolType

CHAIN_ID = ChainId.cross_chain
PRIMARY_CHAIN_ID = ChainId.ethereum
HYPERCORE_CHAIN_ID = ChainId.hypercore


# We define our main trading universe,
# and then Ethereum mainnet as a validation set
match CHAIN_ID:

    case ChainId.cross_chain:
        # Special backtest for pairs across all chains what a cross chain strategy would look lijke

        EXCHANGES = ("uniswap-v2", "uniswap-v3")
        SUPPORTING_PAIRS = [
            (ChainId.arbitrum, "uniswap-v3", "WETH", "USDC", 0.0005),
            (ChainId.ethereum, "uniswap-v3", "WETH", "USDC", 0.0005),
            (ChainId.ethereum, "uniswap-v3", "WBTC", "USDC", 0.003),
        ]
        LENDING_RESERVES = None
        PREFERRED_STABLECOIN = USDC_NATIVE_TOKEN[PRIMARY_CHAIN_ID].lower()


        # Dynamic Hypercore vault universe — fetched from Trading Strategy API
        # and cached by parameters in ~/.cache/indicators/
        # Clear cache with /clear-backtesting-cache skill
        from getting_started.hyperliquid_vault_universe import build_hyperliquid_vault_universe
        VAULTS = build_hyperliquid_vault_universe(
            min_tvl=10_000,
            min_age=0.15,
        )

        # Exclude Euro vaults, etc.
        # ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC", "USDT", "USDC.e"}
        ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC", "USDT", "USDC.e", "crvUSD", "USD₮0", "USDt", "USDT0", "USDS"}

        BENCHMARK_PAIRS = [
            (ChainId.ethereum, "uniswap-v3", "WBTC", "USDC", 0.003),
        ]
    case ChainId.arbitrum:

        EXCHANGES = ("uniswap-v2", "uniswap-v3")
        SUPPORTING_PAIRS = [
            (ChainId.arbitrum, "uniswap-v3", "WETH", "USDC", 0.0005),
        ]
        LENDING_RESERVES = None
        PREFERRED_STABLECOIN = USDC_NATIVE_TOKEN[PRIMARY_CHAIN_ID].lower()

        VAULTS = [
            # Arbitrum only
            (ChainId.arbitrum, "0x58bfc95a864e18e8f3041d2fcd3418f48393fe6a"),  # Plutus Hedge Token
            (ChainId.arbitrum, "0x959f3807f0aa7921e18c78b00b2819ba91e52fef"),  # gmUSDC
            (ChainId.arbitrum, "0xd3443ee1e91af28e5fb858fbd0d72a63ba8046e0"),  # gTrade (Gains) USDC
            (ChainId.arbitrum, "0x75288264fdfea8ce68e6d852696ab1ce2f3e5004"),  # Hype++
            (ChainId.arbitrum, "0x4b6f1c9e5d470b97181786b26da0d0945a7cf027"),  # Hypertrim USDC
            (ChainId.arbitrum, "0x0b2b2b2076d95dda7817e785989fe353fe955ef9"),  # Staked USDai
            (ChainId.arbitrum, "0x64ca76e2525fc6ab2179300c15e343d73e42f958"),  # Clearstar high yielsd USDC
            (ChainId.arbitrum, "0x7e97fa6893871a2751b5fe961978dccb2c201e65"),  # Gauntlet
            (ChainId.arbitrum, "0x1a996cb54bb95462040408c06122d45d6cdb6096"),  # Fluid
            (ChainId.arbitrum, "0xa91267a25939b2b0f046013fbf9597008f7f014b"),  # IPOR USDC Arbirum optimise
            (ChainId.arbitrum, "0x05d28a86e057364f6ad1a88944297e58fc6160b3"),  # Euler Arbitrum Yield USDC
            (ChainId.arbitrum, "0xc8248953429d707c6a2815653eca89846ffaa63b"),  # Curve LLAMMA asdCRV / crvUSD
            (ChainId.arbitrum, "0xf63b7f49b4f5dc5d0e7e583cfd79dc64e646320c"),  # Auto finance Tokemak ARB/USDC
            (ChainId.arbitrum, "0xeeaf2ccb73a01deb38eca2947d963d64cfde6a32"),  # Curve LLAMMA CRV / crvUSD
            (ChainId.arbitrum, "0xe5d6eb448ac5a762c1ebe8cd1692b9cd08025176"),  # DAMM stablecoin fund
        ]

        # Exclude Euro vaults, etc.
        # ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC", "USDT", "USDC.e"}
        ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC"}
    
    case ChainId.base:
        EXCHANGES = ("uniswap-v2", "uniswap-v3", "aerodrome")
        SUPPORTING_PAIRS = [
            (ChainId.base, "uniswap-v3", "WETH", "USDC", 0.0005),
        ]
        LENDING_RESERVES = None
        PREFERRED_STABLECOIN = USDC_NATIVE_TOKEN[ChainId.base].lower()

        VAULTS = [
            (ChainId.base, "0x13cd7cf42ccbaca8cd97e7f09572b6ea0de1097b"), # USDC-TIBBIR shares
        ]

        ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC"}


print(f"Chain universe: {CHAIN_ID}")
print(f"Vaults count: {len(VAULTS)}")
print(f"Allowed denomination tokens: {ALLOWED_VAULT_DENOMINATION_TOKENS}")

# Parameters

- Collection of parameters used in the calculations

In [ ]:
import datetime

import pandas as pd

from tradingstrategy.timebucket import TimeBucket
from tradeexecutor.strategy.cycle import CycleDuration
from tradeexecutor.strategy.parameters import StrategyParameters
from skopt.space import Categorical
from tradeexecutor.strategy.default_routing_options import TradeRouting

from tradeexecutor.utils.jupyter_notebook_name import get_notebook_id

class Parameters:

    id = get_notebook_id(globals())

    # We trade 1h candle
    candle_time_bucket = TimeBucket.d1
    cycle_duration = CycleDuration.cycle_1d

    chain_id = CHAIN_ID
    primary_chain_id = PRIMARY_CHAIN_ID
    exchanges = EXCHANGES

    #
    # Basket size, risk and balancing parameters.
    #
    max_assets_in_portfolio = 30
    allocation = 0.98
    individual_rebalance_min_threshold_usd = 50.0
    sell_rebalance_min_threshold = 10.0
    per_position_cap_of_pool = 0.20
    max_concentration = 0.25
    min_portfolio_weight = 0.0050

    # Gate signal design, fixed close to the NB68/NB70 winner
    gate_signal_variant = "bayesian_sterling"
    gate_mode = "positive_only"
    gate_quantile = 0.7
    gate_top_k = 8

    # Rank-composite design
    rank_feature_set = Categorical([
        "returns + sharpe + age + tvl",
        "returns + sharpe + drawdown + age + tvl",
        "returns + sterling + drawdown + age + tvl",
    ])

    rank_weight_scheme = Categorical([
        "equal_feature_weights",
        "performance_heavy",
        "credibility_heavy",
    ])

    selection_mode = Categorical([
        "top_k",
        "top_quantile",
    ])
    selection_k = Categorical([5, 8, 12])
    selection_quantile = Categorical([0.2, 0.3, 0.5])

    weight_function = Categorical([
        "weight_equal",
        "weight_blend_0.5",
        "weight_softmax_2.0",
    ])

    # Gate hyperparameters
    volatility_window = Categorical([120, 150])
    bayesian_halflife = Categorical([30, 60])
    sterling_constant = Categorical([0.10, 0.20])
    calmar_signal_transform = "bayesian_credibility"

    waterfall = False
    min_tvl = 25_000
    min_age = 0.3

    #
    # Backtesting only
    # No meaningful Hyperliquid vault data/activity before August 2025
    #
    backtest_start = datetime.datetime(2025, 8, 1)
    backtest_end = datetime.datetime(2026, 3, 11)
    initial_cash = 100_000

    #
    # Live only
    #
    routing = TradeRouting.default
    required_history_period = datetime.timedelta(days=60*2)
    slippage_tolerance = 0.0060
    assummed_liquidity_when_data_missings = 10_000


parameters = StrategyParameters.from_class(Parameters)

from tradeexecutor.strategy.parameters import display_parameters
display_parameters(parameters)


# Trading pairs and market data

- This creates the strategy universe containing pair metadata and their prices
- The universe is "masked" by simply selecting pairs on the predefined pairs list

In [ ]:

from pathlib import Path
from typing import Callable
from tradingstrategy.pair import PandasPairUniverse


from eth_defi.vault.vaultdb import DEFAULT_VAULT_DATABASE, DEFAULT_RAW_PRICE_DATABASE


from tradingstrategy.utils.token_filter import filter_for_selected_pairs
from tradingstrategy.alternative_data.vault import load_vault_database


from tradeexecutor.strategy.trading_strategy_universe import TradingStrategyUniverse
from tradeexecutor.strategy.execution_context import ExecutionContext, notebook_execution_context
from tradeexecutor.strategy.universe_model import UniverseOptions
from tradeexecutor.strategy.trading_strategy_universe import TradingStrategyUniverse, load_partial_data
from tradeexecutor.strategy.execution_context import ExecutionContext, notebook_execution_context
from tradeexecutor.strategy.universe_model import UniverseOptions
from tradeexecutor.analysis.pair import display_strategy_universe
from tradeexecutor.strategy.pandas_trader.trading_universe_input import CreateTradingUniverseInput
from tradeexecutor.analysis.vault import display_vaults



# Hack to support vault data exposure to live trading universe creation
from dotenv import load_dotenv
load_dotenv(override=True)  # Loads variables from .env file


def create_trading_universe(
    input: CreateTradingUniverseInput,
) -> TradingStrategyUniverse:
    """Create the trading universe.

    - Load Trading Strategy full pairs dataset

    - Load built-in Coingecko top 1000 dataset

    - Get all DEX tokens for a certain Coigecko category

    - Load OHCLV data for these pairs

    - Load also BTC and ETH price data to be used as a benchmark
    """

    execution_context = input.execution_context
    client = input.client
    timestamp = input.timestamp
    parameters = input.parameters or Parameters  # Some CLI commands do not support yet passing this
    universe_options = input.universe_options

    if execution_context.live_trading:
        # Live trading, send strategy universe formation details
        # to logs
        debug_printer = logger.info
    else:
        # Jupyter notebook inline output
        debug_printer = print

    chain_id = parameters.primary_chain_id

    debug_printer(f"Preparing trading universe on chain {chain_id.get_name()}")

    # Pull out our benchmark pairs ids.
    # We need to construct pair universe object for the symbolic lookup.
    # TODO: PandasPairUniverse(buidl_index=True) - speed this up by skipping index building
    all_pairs_df = client.fetch_pair_universe().to_pandas()
    pairs_df= filter_for_selected_pairs(
        all_pairs_df,
        SUPPORTING_PAIRS,
    )    

    debug_printer(f"We have total {len(all_pairs_df)} pairs in dataset and going to use {len(pairs_df)} pairs for the strategy")

    # Check which vaults we can include based on allowed deposit tokens for this backtest
    # Use local vault database (~/.tradingstrategy/vaults/) instead of
    # the bundled one, as the bundled one may not have all chains
    vault_universe = load_vault_database(path=DEFAULT_VAULT_DATABASE)
    total_vaults = vault_universe.get_vault_count()
    vault_universe = vault_universe.limit_to_vaults(VAULTS, check_all_vaults_found=False)
    vault_universe = vault_universe.limit_to_denomination(ALLOWED_VAULT_DENOMINATION_TOKENS, check_all_vaults_found=True)
    debug_printer(f"Loaded total {vault_universe.get_vault_count()} vaults from the total of {total_vaults} in vault database, source vaults count: {len(VAULTS)}")

    # Default vault data bundle path for backtesting
    vault_bundled_price_data = DEFAULT_RAW_PRICE_DATABASE
    debug_printer(f"Using vault price data for backtesting from {vault_bundled_price_data}")

    dataset = load_partial_data(
        client=client,
        time_bucket=parameters.candle_time_bucket,
        pairs=pairs_df,
        execution_context=execution_context,
        universe_options=universe_options,
        liquidity=True,
        liquidity_time_bucket=TimeBucket.d1,
        lending_reserves=LENDING_RESERVES,
        vaults=vault_universe,
        vault_bundled_price_data=vault_bundled_price_data,
        check_all_vaults_found=True,
    )

    debug_printer("Creating strategy universe with price feeds and vaults")
    strategy_universe = TradingStrategyUniverse.create_from_dataset(
        dataset,
        reserve_asset=PREFERRED_STABLECOIN,
        forward_fill=True,  # We got very gappy data from low liquid DEX coins
        forward_fill_until=timestamp,
        primary_chain=parameters.primary_chain_id
    )
    return strategy_universe


universe_input = CreateTradingUniverseInput(
    execution_context=notebook_execution_context,
    client=client,
    timestamp=None,
    parameters=parameters,
    universe_options=UniverseOptions.from_strategy_parameters_class(Parameters, notebook_execution_context),
    execution_model=None,
)

strategy_universe = create_trading_universe(universe_input)


print("Universe creation done")

df = display_strategy_universe(
    strategy_universe, 
    sort_key="base",
    sort_numeric=False,
    limit=75,
    show_token_risk=True,
    show_tokensniffer=True,
)

df = df.head(3)

from tradeexecutor.utils.notebook import display_dataframe_with_html

display_dataframe_with_html(df)

# Indicators

- Precalculate indicators used by the strategy

In [ ]:
import pandas as pd
from IPython.display import HTML
import pandas_ta

from tradeexecutor.strategy.pandas_trader.indicator import IndicatorSource
from tradeexecutor.strategy.trading_strategy_universe import TradingStrategyUniverse
from tradeexecutor.strategy.pandas_trader.indicator import calculate_and_load_indicators_inline
from tradeexecutor.strategy.pandas_trader.indicator import IndicatorDependencyResolver
from tradeexecutor.state.types import USDollarAmount
from tradeexecutor.strategy.pandas_trader.indicator_decorator import IndicatorRegistry
from tradeexecutor.analysis.indicator import display_indicators
from tradeexecutor.state.identifier import TradingPairIdentifier


indicators = IndicatorRegistry()

empty_series = pd.Series([], index=pd.DatetimeIndex([]))


def _pad_daily_returns(close: pd.Series, volatility_window: int) -> pd.Series:
    """Helper: compute daily returns and pad with zeros at the start.

    Shared by all ratio indicators to ensure consistent treatment of
    young vaults / missing history.
    """
    daily_returns = close.pct_change().fillna(0)
    pad_index = pd.date_range(
        end=daily_returns.index[0] - pd.Timedelta(days=1),
        periods=volatility_window,
        freq=daily_returns.index.inferred_freq or "D",
    )
    pad = pd.Series(0.0, index=pad_index)
    return pd.concat([pad, daily_returns])


@indicators.define()
def rolling_returns(
    close: pd.Series,
    volatility_window: int = 180,
) -> pd.Series:
    """Calculate annualised rolling returns.

    - Uses volatility_window as the lookback period
    - Requires minimum 7 days of data
    - Annualises all returns to be comparable across different data lengths
    - Simplest possible signal: no risk adjustment whatsoever
    """
    min_periods = 7

    first_price = close.rolling(
        window=volatility_window,
        min_periods=min_periods,
    ).apply(lambda x: x[0], raw=True)

    actual_days = close.rolling(
        window=volatility_window,
        min_periods=min_periods,
    ).apply(lambda x: len(x), raw=True)

    period_return = close / first_price - 1
    annualised = (1 + period_return) ** (365 / actual_days) - 1
    return annualised


@indicators.define()
def rolling_sharpe(
    close: pd.Series,
    volatility_window: int = 180,
) -> pd.Series:
    """Calculate rolling Sharpe ratio.

    - Annualised return / annualised volatility
    - Uses volatility_window for lookback period
    - Requires minimum 14 days of data
    - Risk-adjusted but without the drawdown complexity of Calmar
    """
    min_periods = 14
    daily_returns = close.pct_change()

    rolling_mean = daily_returns.rolling(
        window=volatility_window,
        min_periods=min_periods,
    ).mean()

    rolling_std = daily_returns.rolling(
        window=volatility_window,
        min_periods=min_periods,
    ).std()

    # Annualise: mean * 365 / (std * sqrt(365)) = mean * sqrt(365) / std
    sharpe = (rolling_mean * 365) / (rolling_std * (365 ** 0.5))
    return sharpe


@indicators.define()
def rolling_calmar(
    close: pd.Series,
    volatility_window: int = 180,
) -> pd.Series:
    """Calculate rolling Calmar ratio.

    - Annualised return / maximum drawdown over the lookback window
    - Uses volatility_window for lookback period
    - Pads start with zero returns so missing history counts as flat days,
      preventing extreme spikes for young vaults
    - NaN returns (from first pct_change or price gaps) treated as flat (0)
    """
    daily_returns = _pad_daily_returns(close, volatility_window)

    rolling_mean = daily_returns.rolling(
        window=volatility_window,
        min_periods=volatility_window,
    ).mean()

    def _max_drawdown(window):
        cumulative = (1 + window).cumprod()
        peak = cumulative.cummax()
        dd = (cumulative / peak - 1).min()
        return max(abs(dd), 1e-4)

    rolling_mdd = daily_returns.rolling(
        window=volatility_window,
        min_periods=volatility_window,
    ).apply(_max_drawdown, raw=False)

    calmar = (rolling_mean * 365) / rolling_mdd

    # Trim padding back to original index
    calmar = calmar.reindex(close.index)
    return calmar


@indicators.define()
def rolling_sterling_floor(
    close: pd.Series,
    volatility_window: int = 180,
    sterling_constant: float = 0.10,
) -> pd.Series:
    """Rolling Sterling-floor ratio: annualised return / (max_drawdown + C).

    Adds a constant C to the denominator, following Sterling (1981).
    This prevents near-zero drawdowns from producing extreme ratios:
    - Vault with 0% drawdown -> ratio = return / C (bounded)
    - Vault with 50% drawdown -> ratio = return / (50% + C) (constant barely matters)

    :param sterling_constant:
        Floor added to max drawdown in the denominator.
        Sterling (1981) used 10% to match T-bill rates of the era.
        Today the value is arbitrary but serves as numerical stability floor.
    """
    daily_returns = _pad_daily_returns(close, volatility_window)

    rolling_mean = daily_returns.rolling(
        window=volatility_window,
        min_periods=volatility_window,
    ).mean()

    def _max_drawdown(window):
        cumulative = (1 + window).cumprod()
        peak = cumulative.cummax()
        dd = (cumulative / peak - 1).min()
        return abs(dd)

    rolling_mdd = daily_returns.rolling(
        window=volatility_window,
        min_periods=volatility_window,
    ).apply(_max_drawdown, raw=False)

    sterling = (rolling_mean * 365) / (rolling_mdd + sterling_constant)

    sterling = sterling.reindex(close.index)
    return sterling


@indicators.define()
def drawdown_from_peak(
    close: pd.Series,
) -> pd.Series:
    """Calculate running drawdown from share price peak.

    Returns negative values, e.g. -0.20 means vault is 20% below ATH.
    """
    cummax = close.cummax()
    drawdown = close / cummax - 1
    return drawdown


@indicators.define(dependencies=(drawdown_from_peak,), source=IndicatorSource.dependencies_only_per_pair)
def drawdown_duration(
    pair: TradingPairIdentifier,
    dependency_resolver: IndicatorDependencyResolver,
) -> pd.Series:
    """How many consecutive periods the vault has stayed below peak."""
    drawdown = dependency_resolver.get_indicator_data("drawdown_from_peak", pair=pair)
    below_peak = drawdown < 0
    duration = below_peak.astype(int)
    for idx in range(1, len(duration)):
        if below_peak.iloc[idx]:
            duration.iloc[idx] = duration.iloc[idx - 1] + 1
    return duration.astype(float)


def cross_sectional_rank_spread(rank_df: pd.DataFrame) -> pd.DataFrame:
    """Calculate cross-sectional spread diagnostics for rank-based scores."""
    return pd.DataFrame({
        "std": rank_df.std(axis=1),
        "p90_minus_p10": rank_df.quantile(0.9, axis=1) - rank_df.quantile(0.1, axis=1),
        "top1_minus_median": rank_df.max(axis=1) - rank_df.median(axis=1),
    })


@indicators.define(
    dependencies=(rolling_calmar, rolling_sterling_floor, drawdown_from_peak),
    source=IndicatorSource.dependencies_only_per_pair,
)
def rolling_calmar_transformed(
    pair: TradingPairIdentifier,
    dependency_resolver: IndicatorDependencyResolver,
    volatility_window: int = 180,
    calmar_signal_transform: str = "raw",
    bayesian_halflife: int = 30,
    sterling_constant: float = 0.10,
) -> pd.Series:
    """Apply bayesian credibility signal transform.

    Bayesian shrinkage toward cross-sectional log-Sterling mean,
    weighted by track record length via bayesian_halflife parameter.
    """
    import numpy as np

    calmar = dependency_resolver.get_indicator_data(
        "rolling_calmar", pair=pair,
        parameters={"volatility_window": volatility_window},
    )

    if calmar_signal_transform == "bayesian_credibility":
        # Bayesian shrinkage: w_T * base + (1 - w_T) * prior
        # base = log_sterling for this pair
        # prior = cross-sectional mean of log_sterling
        # w_T = T / (T + H), H = bayesian_halflife parameter
        BAYESIAN_HALFLIFE = bayesian_halflife

        sterling = dependency_resolver.get_indicator_data(
            "rolling_sterling_floor", pair=pair,
            parameters={"volatility_window": volatility_window, "sterling_constant": sterling_constant},
        )
        base = np.log1p(sterling.clip(lower=0))

        # Cross-sectional prior: mean of log_sterling across all vaults
        all_sterling = dependency_resolver.get_indicator_data_pairs_combined(
            "rolling_sterling_floor",
            parameters={"volatility_window": volatility_window, "sterling_constant": sterling_constant},
        )
        sterling_df = all_sterling.unstack(level="pair_id")
        log_sterling_df = np.log1p(sterling_df.clip(lower=0))
        prior = log_sterling_df.mean(axis=1)

        # Data age: count of non-NaN calmar values as proxy for real data
        real_days = (~calmar.isna()).cumsum().astype(float)
        w_t = real_days / (real_days + BAYESIAN_HALFLIFE)

        return w_t * base + (1 - w_t) * prior

    # Fallback
    return calmar

@indicators.define(source=IndicatorSource.tvl)
def tvl(
    close: pd.Series,
    execution_context: ExecutionContext,
    timestamp: pd.Timestamp,
) -> pd.Series:
    """Get TVL series for a pair.

    - Because TVL data is 1d and we use 1h everywhere else, we need to forward fill

    - Use previous hourly close as the value
    """
    if execution_context.live_trading:
        # TVL is daily data.
        # We need to forward fill until the current hour.
        # Use our special ff function.        
        assert isinstance(timestamp, pd.Timestamp), f"Live trading needs forward-fill end time, we got {timestamp}"
        from tradingstrategy.utils.forward_fill import forward_fill
        df = pd.DataFrame({"close": close})
        df_ff = forward_fill(
            df,
            Parameters.candle_time_bucket.to_frequency(),
            columns=("close",),
            forward_fill_until=timestamp,
        )
        series = df_ff["close"]
        return series
    else:
        return close.resample("1h").ffill()


@indicators.define()
def age(
    close: pd.Series,
) -> pd.Series:
    """Calculate vault age in years at each timestamp.

    Age is measured from the first available price data point.
    """
    inception = close.index[0]
    age_years = (close.index - inception) / pd.Timedelta(days=365.25)
    return pd.Series(age_years, index=close.index)






@indicators.define(dependencies=(tvl,), source=IndicatorSource.dependencies_only_universe)
def tvl_inclusion_criteria(   
    min_tvl: USDollarAmount,
    dependency_resolver: IndicatorDependencyResolver,
) -> pd.Series:
    """The pair must have min XX,XXX USD one-sided TVL to be included.

    - If the Uniswap pool does not have enough ETH or USDC deposited, skip the pair as a scam

    :return:
        Series where each timestamp is a list of pair ids meeting the criteria at that timestamp
    """
    
    series = dependency_resolver.get_indicator_data_pairs_combined(tvl)
    mask = series >= min_tvl
    # Turn to a series of lists
    mask_true_values_only = mask[mask == True]
    series = mask_true_values_only.groupby(level='timestamp').apply(lambda x: x.index.get_level_values('pair_id').tolist())
    return series


@indicators.define(dependencies=(age,), source=IndicatorSource.dependencies_only_universe)
def age_inclusion_criteria(
    min_age: float,
    dependency_resolver: IndicatorDependencyResolver,
) -> pd.Series:
    """The pair must be at least min_age years old to be included.

    :return:
        Series where each timestamp is a list of pair ids meeting the age criteria at that timestamp
    """
    series = dependency_resolver.get_indicator_data_pairs_combined(age)
    mask = series >= min_age
    mask_true_values_only = mask[mask == True]
    series = mask_true_values_only.groupby(level='timestamp').apply(lambda x: x.index.get_level_values('pair_id').tolist())
    return series


@indicators.define(
    source=IndicatorSource.strategy_universe
)
def trading_availability_criteria(
    strategy_universe: TradingStrategyUniverse,
) -> pd.Series:
    """Is pair tradeable at each hour.

    - The pair has a price candle at that
    - Mitigates very corner case issues that TVL/liquidity data is per-day whileas price data is natively per 1h
      and the strategy inclusion criteria may include pair too early hour based on TVL only,
      leading to a failed attempt to rebalance in a backtest
    - Only relevant for backtesting issues if we make an unlucky trade on the starting date
      of trading pair listing

    :return:
        Series with with index (timestamp) and values (list of pair ids trading at that hour)
    """
    # Trading pair availability is defined if there is a open candle in the index for it.
    # Because candle data is forward filled, we should not have any gaps in the index.
    candle_series = strategy_universe.data_universe.candles.df["open"]
    pairs_per_timestamp = candle_series.groupby(level='timestamp').apply(lambda x: x.index.get_level_values('pair_id').tolist())
    return pairs_per_timestamp


@indicators.define(
    dependencies=[
        tvl_inclusion_criteria,
        trading_availability_criteria,
        age_inclusion_criteria,
    ],
    source=IndicatorSource.strategy_universe
)
def inclusion_criteria(
    strategy_universe: TradingStrategyUniverse,
    min_tvl: USDollarAmount,
    min_age: float,
    dependency_resolver: IndicatorDependencyResolver
) -> pd.Series:
    """Pairs meeting all of our inclusion criteria.

    - Give the tradeable pair set for each timestamp

    :return:
        Series where index is timestamp and each cell is a list of pair ids matching our inclusion criteria at that moment
    """

    # Filter out benchmark pairs like WETH in the tradeable pair set
    benchmark_pair_ids = set(strategy_universe.get_pair_by_human_description(desc).internal_id for desc in SUPPORTING_PAIRS)

    tvl_series = dependency_resolver.get_indicator_data(
        tvl_inclusion_criteria,
        parameters={
            "min_tvl": min_tvl,
        },
    )

    trading_availability_series = dependency_resolver.get_indicator_data(trading_availability_criteria)

    age_series = dependency_resolver.get_indicator_data(
        age_inclusion_criteria,
        parameters={
            "min_age": min_age,
        },
    )

    #
    # Process all pair ids as a set and the final inclusion
    # criteria is union of all sub-criterias
    #

    df = pd.DataFrame({
        "tvl_pair_ids": tvl_series,
        "trading_availability_pair_ids": trading_availability_series,
        "age_pair_ids": age_series,
    })

    # https://stackoverflow.com/questions/33199193/how-to-fill-dataframe-nan-values-with-empty-list-in-pandas
    df = df.fillna("").apply(list)

    def _combine_criteria(row):
        final_set = set(row["tvl_pair_ids"]) & \
                    set(row["trading_availability_pair_ids"]) & \
                    set(row["age_pair_ids"])
        return final_set - benchmark_pair_ids

    union_criteria = df.apply(_combine_criteria, axis=1)

    # Inclusion criteria data can be spotty at the beginning when there is only 0 or 1 pairs trading,
    # so we need to fill gaps to 0
    full_index = pd.date_range(
        start=union_criteria.index.min(),
        end=union_criteria.index.max(),
        freq=Parameters.candle_time_bucket.to_frequency(),
    )
    reindexed = union_criteria.reindex(full_index, fill_value=[])
    return reindexed


@indicators.define(dependencies=(tvl_inclusion_criteria,), source=IndicatorSource.dependencies_only_universe)
def tvl_included_pair_count(
        min_tvl: USDollarAmount,
        dependency_resolver: IndicatorDependencyResolver
) -> pd.Series:
    """Calculate number of pairs in meeting volatility criteria on each timestamp"""
    series = dependency_resolver.get_indicator_data(
        tvl_inclusion_criteria,
        parameters={"min_tvl": min_tvl},
    )
    series = series.apply(len)

    # TVL data can be spotty at the beginning when there is only 0 or 1 pairs trading,
    # so we need to fill gaps to 0
    full_index = pd.date_range(
        start=series.index.min(),
        end=series.index.max(),
        freq=Parameters.candle_time_bucket.to_frequency(),
    )
    # Reindex and fill NaN with zeros
    reindexed = series.reindex(full_index, fill_value=0)
    return reindexed


@indicators.define(dependencies=(age_inclusion_criteria,), source=IndicatorSource.dependencies_only_universe)
def age_included_pair_count(
        min_age: float,
        dependency_resolver: IndicatorDependencyResolver
) -> pd.Series:
    """Calculate number of pairs meeting age criteria on each timestamp."""
    series = dependency_resolver.get_indicator_data(
        age_inclusion_criteria,
        parameters={
            "min_age": min_age,
        },
    )
    series = series.apply(len)

    full_index = pd.date_range(
        start=series.index.min(),
        end=series.index.max(),
        freq=Parameters.candle_time_bucket.to_frequency(),
    )
    reindexed = series.reindex(full_index, fill_value=0)
    return reindexed


@indicators.define(dependencies=(inclusion_criteria,), source=IndicatorSource.dependencies_only_universe)
def all_criteria_included_pair_count(
    min_tvl: USDollarAmount,
    min_age: float,
    dependency_resolver: IndicatorDependencyResolver
) -> pd.Series:
    """Series where each timestamp is the list of pairs meeting all inclusion criteria.

    :return:
        Series with pair count for each timestamp
    """
    series = dependency_resolver.get_indicator_data(
        "inclusion_criteria",
        parameters={
            "min_tvl": min_tvl,
            "min_age": min_age,
        },
    )
    return series.apply(len)


@indicators.define(source=IndicatorSource.strategy_universe)
def trading_pair_count(
    strategy_universe: TradingStrategyUniverse,
) -> pd.Series:
    """Get number of pairs that trade at each timestamp.

    - Pair must have had at least one candle before the timestamp to be included

    - Exclude benchmarks pairs we do not trade

    :return:
        Series with pair count for each timestamp
    """

    benchmark_pair_ids = {strategy_universe.get_pair_by_human_description(desc).internal_id for desc in SUPPORTING_PAIRS}

    # Get pair_id, timestamp -> timestamp, pair_id index
    series = strategy_universe.data_universe.candles.df["open"]    
    swap_index = series.index.swaplevel(0, 1)

    seen_pairs = set()
    seen_data = {}

    for timestamp, pair_id in swap_index:
        if pair_id in benchmark_pair_ids:
            continue
        seen_pairs.add(pair_id)
        seen_data [timestamp] = len(seen_pairs)

    series = pd.Series(seen_data.values(), index=list(seen_data.keys()))
    return series


display_indicators(indicators)


# Time range for backtest

- Choose the backtesting time range
- Start when we have enough assets (`Parameters.min_asset_universe`) in our asset universe to form the first basket

In [ ]:
backtest_start = Parameters.backtest_start
backtest_end = Parameters.backtest_end

print(f"Time range is {backtest_start} - {backtest_end}")


# Algorithm and backtest

- Run the backtest

In [ ]:
import numpy as np

from tradeexecutor.backtest.backtest_runner import run_backtest_inline
from tradeexecutor.strategy.alpha_model import AlphaModel
from tradeexecutor.state.trade import TradeExecution
from tradeexecutor.strategy.pandas_trader.strategy_input import StrategyInput, IndicatorDataNotFoundWithinDataTolerance
from tradeexecutor.state.visualisation import PlotKind
from tradeexecutor.backtest.backtest_runner import run_backtest_inline
from tradeexecutor.strategy.tvl_size_risk import USDTVLSizeRiskModel
from tradeexecutor.strategy.weighting import weight_equal, weight_by_softmax, weight_by_blend
from tradeexecutor.utils.dedent import dedent_any
from tradeexecutor.strategy.pandas_trader.yield_manager import YieldManager, YieldRuleset, YieldWeightingRule, YieldDecisionInput
from tradeexecutor.strategy.execution_context import ExecutionContext, ExecutionMode
from getting_started.curator import is_quarantined


def _rank_percentile(series: pd.Series, ascending: bool) -> pd.Series:
    """Convert raw feature values to cross-sectional percentile ranks in [0, 1]."""
    if len(series) == 0:
        return series
    ranked = series.rank(method="average", ascending=ascending, pct=True)
    return ranked.clip(lower=0.0, upper=1.0)


def _build_rank_weights(rank_feature_set: str, rank_weight_scheme: str) -> dict[str, float]:
    """Map rank feature set and weight scheme to final feature weights."""
    feature_sets = {
        "returns + sharpe + age + tvl": ["rolling_returns", "rolling_sharpe", "age", "tvl"],
        "returns + sharpe + drawdown + age + tvl": ["rolling_returns", "rolling_sharpe", "drawdown_from_peak", "age", "tvl"],
        "returns + sterling + drawdown + age + tvl": ["rolling_returns", "rolling_sterling_floor", "drawdown_from_peak", "age", "tvl"],
    }
    features = feature_sets[rank_feature_set]

    performance_features = {"rolling_returns", "rolling_sharpe", "rolling_sterling_floor"}
    credibility_features = {"age", "tvl", "drawdown_from_peak", "drawdown_duration"}

    if rank_weight_scheme == "performance_heavy":
        performance_budget = 0.60
        credibility_budget = 0.40
    elif rank_weight_scheme == "credibility_heavy":
        performance_budget = 0.40
        credibility_budget = 0.60
    else:
        return {feature: 1.0 / len(features) for feature in features}

    selected_performance = [feature for feature in features if feature in performance_features]
    selected_credibility = [feature for feature in features if feature in credibility_features]

    weights = {}
    if selected_performance:
        share = performance_budget / len(selected_performance)
        for feature in selected_performance:
            weights[feature] = share
    if selected_credibility:
        share = credibility_budget / len(selected_credibility)
        for feature in selected_credibility:
            weights[feature] = share

    total = sum(weights.values())
    return {feature: weight / total for feature, weight in weights.items()}


def decide_trades(
    input: StrategyInput
) -> list[TradeExecution]:
    """For each strategy tick, generate the list of trades.

    Rank-composite approach:
    1. Bayesian sterling gate decides which vaults are eligible
    2. Gated survivors are ranked ordinally on a composite score
    3. Selection keeps only the strongest ranked survivors
    4. Weighting compares equal weight against mild tilts
    """
    parameters = input.parameters
    position_manager = input.get_position_manager()
    state = input.state
    timestamp = input.timestamp
    indicators = input.indicators
    strategy_universe = input.strategy_universe

    portfolio = position_manager.get_current_portfolio()
    equity = portfolio.get_total_equity()

    if input.execution_context.mode == ExecutionMode.backtesting:
        if equity < parameters.initial_cash * 0.10:
            return []

    alpha_model = AlphaModel(
        timestamp,
        close_position_weight_epsilon=parameters.min_portfolio_weight,
    )

    tvl_included_pair_count = indicators.get_indicator_value("tvl_included_pair_count")

    included_pairs = indicators.get_indicator_value(
        "inclusion_criteria",
        na_conversion=False,
    )
    if included_pairs is None:
        included_pairs = []

    candidates = []
    gate_filtered = 0
    skipped_pairs = 0

    feature_directions = {
        "rolling_returns": False,
        "rolling_sharpe": False,
        "rolling_sterling_floor": False,
        "age": False,
        "tvl": False,
        "drawdown_from_peak": False,
        "drawdown_duration": True,
    }

    rank_weights = _build_rank_weights(parameters.rank_feature_set, parameters.rank_weight_scheme)
    selected_features = list(rank_weights.keys())

    for pair_id in included_pairs:
        pair = strategy_universe.get_pair_by_id(pair_id)

        if not state.is_good_pair(pair):
            skipped_pairs += 1
            continue

        if is_quarantined(pair.pool_address, timestamp):
            skipped_pairs += 1
            continue

        gate_signal = indicators.get_indicator_value("rolling_calmar_transformed", pair=pair)
        if gate_signal is None or pd.isna(gate_signal) or not np.isfinite(gate_signal):
            skipped_pairs += 1
            continue

        if gate_signal <= 0:
            gate_filtered += 1
            continue

        feature_values = {}
        missing_feature = False
        for feature_name in selected_features:
            value = indicators.get_indicator_value(feature_name, pair=pair)
            if value is None or pd.isna(value) or not np.isfinite(value):
                missing_feature = True
                break
            feature_values[feature_name] = float(value)

        if missing_feature:
            skipped_pairs += 1
            continue

        candidates.append({
            "pair": pair,
            "gate_signal": float(gate_signal),
            **feature_values,
        })

    if not candidates:
        return []

    candidate_df = pd.DataFrame(candidates)
    rank_columns = []
    for feature_name in selected_features:
        ascending = feature_directions[feature_name]
        rank_column = f"rank_{feature_name}"
        candidate_df[rank_column] = _rank_percentile(candidate_df[feature_name], ascending=ascending)
        rank_columns.append(rank_column)

    candidate_df["rank_composite_signal"] = 0.0
    for feature_name, weight in rank_weights.items():
        candidate_df["rank_composite_signal"] += candidate_df[f"rank_{feature_name}"] * weight

    candidate_df = candidate_df.replace([np.inf, -np.inf], np.nan)
    candidate_df = candidate_df.dropna(subset=["rank_composite_signal"])

    if parameters.selection_mode == "top_quantile":
        threshold = candidate_df["rank_composite_signal"].quantile(float(parameters.selection_quantile))
        selected_df = candidate_df[candidate_df["rank_composite_signal"] >= threshold].copy()
    else:
        selected_df = candidate_df.nlargest(int(parameters.selection_k), "rank_composite_signal").copy()

    signal_count = 0
    for _, candidate in selected_df.iterrows():
        alpha_model.set_signal(candidate["pair"], float(candidate["rank_composite_signal"]))
        signal_count += 1

    portfolio = position_manager.get_current_portfolio()
    equity = portfolio.get_total_equity()
    portfolio_target_value = equity * float(parameters.allocation)

    alpha_model.select_top_signals(count=999)

    weight_func_map = {
        "weight_equal": weight_equal,
        "weight_softmax_2.0": lambda s: weight_by_softmax(s, temperature=2.0),
        "weight_blend_0.5": lambda s: weight_by_blend(s, blend_alpha=0.5),
    }
    weight_func = weight_func_map[parameters.weight_function]
    alpha_model.assign_weights(method=weight_func)

    zero_weight_pairs = [
        pair_id for pair_id, s in alpha_model.signals.items()
        if s.raw_weight == 0.0
    ]
    for pair_id in zero_weight_pairs:
        del alpha_model.signals[pair_id]

    size_risk_model = USDTVLSizeRiskModel(
        pricing_model=input.pricing_model,
        per_position_cap=float(parameters.per_position_cap_of_pool),
        missing_tvl_placeholder_usd=0.0,
    )

    alpha_model.normalise_weights(
        investable_equity=portfolio_target_value,
        size_risk_model=size_risk_model,
        max_weight=float(parameters.max_concentration),
        max_positions=parameters.max_assets_in_portfolio,
        waterfall=parameters.waterfall,
    )

    alpha_model.update_old_weights(
        state.portfolio,
        ignore_credit=False,
    )
    alpha_model.calculate_target_positions(position_manager)

    rebalance_threshold_usd = parameters.individual_rebalance_min_threshold_usd
    assert rebalance_threshold_usd > 0.1, "Safety check tripped - something like wrong with strat code"
    trades = alpha_model.generate_rebalance_trades_and_triggers(
        position_manager,
        min_trade_threshold=rebalance_threshold_usd,
        invidiual_rebalance_min_threshold=parameters.individual_rebalance_min_threshold_usd,
        sell_rebalance_min_threshold=parameters.sell_rebalance_min_threshold,
        execution_context=input.execution_context,
    )

    if input.is_visualisation_enabled():
        try:
            top_signal = next(iter(alpha_model.get_signals_sorted_by_weight()))
            if top_signal.normalised_weight == 0:
                top_signal = None
        except StopIteration:
            top_signal = None

        rebalance_volume = sum(t.get_value() for t in trades)

        report = dedent_any(f"""
        Cycle: #{input.cycle}
        Rebalanced: {'👍' if alpha_model.is_rebalance_triggered() else '👎'}
        Open/about to open positions: {len(state.portfolio.open_positions)}
        Max position value change: {alpha_model.max_position_adjust_usd:,.2f} USD
        Rebalance threshold: {alpha_model.position_adjust_threshold_usd:,.2f} USD
        Trades decided: {len(trades)}
        Pairs total: {strategy_universe.data_universe.pairs.get_count()}
        Pairs meeting inclusion criteria: {len(included_pairs)}
        Pairs meeting TVL inclusion criteria: {tvl_included_pair_count}
        Signals created: {signal_count}
        Filtered by gate: {gate_filtered}
        Skipped before gate: {skipped_pairs}
        Gate signal: {parameters.gate_signal_variant}
        Selection mode: {parameters.selection_mode}
        Rank feature set: {parameters.rank_feature_set}
        Rank weight scheme: {parameters.rank_weight_scheme}
        Weight function: {parameters.weight_function}
        Total equity: {portfolio.get_total_equity():,.2f} USD
        Cash: {position_manager.get_current_cash():,.2f} USD
        Investable equity: {alpha_model.investable_equity:,.2f} USD
        Accepted investable equity: {alpha_model.accepted_investable_equity:,.2f} USD
        Allocated to signals: {alpha_model.get_allocated_value():,.2f} USD
        Discarted allocation because of lack of lit liquidity: {alpha_model.size_risk_discarded_value:,.2f} USD
        Rebalance volume: {rebalance_volume:,.2f} USD
        """)

        if top_signal:
            assert top_signal.position_size_risk
            report += dedent_any(f"""
            Top signal pair: {top_signal.pair.get_ticker()}
            Top signal value: {top_signal.signal}
            Top signal weight: {top_signal.raw_weight}
            Top signal weight (normalised): {top_signal.normalised_weight * 100:.2f} % (got {top_signal.position_size_risk.get_relative_capped_amount() * 100:.2f} % of asked size)
            """)

        for flag, count in alpha_model.get_flag_diagnostics_data().items():
            report += f"Signals with flag {flag.name}: {count}\n"

        state.visualisation.add_message(timestamp, report)
        state.visualisation.set_discardable_data("alpha_model", alpha_model)

    return trades


# Optimiser

- Run optimiser

In [ ]:
import os

# Force terminal/stdout tqdm progress bars during `jupyter execute`.
# Notebook kernels are still IPython, but we want followable TTY output.
os.environ["TQDM_LOGGABLE_FORCE"] = "stdout"

import logging
from tradeexecutor.backtest.optimiser import perform_optimisation
from tradeexecutor.backtest.optimiser import prepare_optimiser_parameters
from tradeexecutor.backtest.optimiser import MinTradeCountFilter
from tradeexecutor.backtest.optimiser_functions import optimise_sharpe, optimise_sortino, optimise_profit, optimise_calmar

# How many Gaussian Process iterations we do
iterations = 10

# What do we optimise for
search_func = optimise_calmar

optimiser_result = perform_optimisation(
    iterations=iterations,
    search_func=search_func,
    decide_trades=decide_trades,
    strategy_universe=strategy_universe,
    parameters=prepare_optimiser_parameters(Parameters),  # Handle scikit-optimise search space
    create_indicators=indicators.create_indicators,
    result_filter=MinTradeCountFilter(50),
    timeout=70*60,    
    batch_size=1,
    # We are searching wacky parameter combinations and
    # some of them lead to buggy strategies,
    # by setting ignore_wallet_errors we just zero out buggy
    # strategies instead of crashing with an exception
    ignore_wallet_errors=True,  
    # Uncomment for diagnostics
    # log_level=logging.INFO,
    max_workers=1,
)

print(f"Optimise completed, optimiser searched {optimiser_result.get_combination_count()} combinations, with {optimiser_result.get_cached_count()} results read directly from cache. and {optimiser_result.get_filtered_count()} filtered results.")
print("Backtests failed with exception:", optimiser_result.get_failed_count())


# Results

- Show the top results of all optimiser iterations in a single table

In [ ]:
from tradeexecutor.analysis.optimiser import analyse_optimiser_result
from tradeexecutor.analysis.grid_search import render_grid_search_result_table


filtered = [r for r in optimiser_result.results if r.filtered]
print(f"Filtering out {len(filtered)} results")

# Optimiser already filtered for min_positions_threshold when doing the optimiser run
df = analyse_optimiser_result(
    optimiser_result,
    max_search_results=300,
    drop_duplicates=False,
)
print(f"Showing the best {len(df)} results")

render_grid_search_result_table(df, calmar=True, sortino=False)
 

# Parameter analysis

## Decision tree visualisation

In [ ]:
from tradeexecutor.analysis.grid_search_parameters import analyse_decision_tree

# Drop NaN rows (from filtered/failed parameter combinations) before decision tree analysis
df_clean = df.dropna(subset=[optimiser_result.target_metric_name])
fig, tree = analyse_decision_tree(df_clean, analysis_metric=optimiser_result.target_metric_name)
fig.show()

## Feature importance analysis

Use a regression model to determine which parameters have the strongest influence.



In [ ]:
from tradeexecutor.analysis.grid_search_parameters import analyse_feature_importance

fig, importances = analyse_feature_importance(df_clean, analysis_metric=optimiser_result.target_metric_name)
fig.show()

## Heatmaps for parameter pairs

In [ ]:
from tradeexecutor.analysis.grid_search_parameters import analyse_parameter_pair_heatmaps
import plotly.graph_objects as go

# Pre-warm Kaleido to avoid deadlock on first fig.show()
# https://github.com/plotly/Kaleido/issues/134
go.Figure().to_image(format="png")

figs = analyse_parameter_pair_heatmaps(df_clean, analysis_metric=optimiser_result.target_metric_name)
for fig in figs:
    fig.show()

# Equity curves

- Visualise equity curves of all grid search results in a single chart

In [ ]:
from tradeexecutor.visual.grid_search_basic import visualise_grid_search_equity_curves
from tradeexecutor.analysis.multi_asset_benchmark import get_benchmark_data
from tradeexecutor.utils.notebook import interactive_plotly_renderer

# Use rank_feature_set for grouping; this is the main cross-section we want to inspect first
top_feature = "rank_feature_set"

# Automatically create BTC and ETH buy and hold benchmark if present
# in the trading universe
benchmark_indexes = get_benchmark_data(
    strategy_universe,
    cumulative_with_initial_cash=Parameters.initial_cash,
)

# Extract GridSearchResult instances from optimiser results
grid_search_results = [r.result for r in optimiser_result.results if not r.filtered]

fig = visualise_grid_search_equity_curves(
    grid_search_results,
    benchmark_indexes=benchmark_indexes,
    log_y=False,
    group_by=top_feature,
    color_mode="discrete",
)

# Override static renderer for this cell so legend items are clickable
with interactive_plotly_renderer():
    fig.show()


# The best candidate equity curve

In [ ]:
from tradeexecutor.visual.grid_search import visualise_single_grid_search_result_benchmark

# GridSearchResult instance that gave the best performance
best_pick = optimiser_result.results[0].result
state = best_pick.hydrate_state()

print(f"The best result found for {search_func} was {best_pick}")

fig = visualise_single_grid_search_result_benchmark(
    best_pick, 
    strategy_universe, 
    initial_cash=Parameters.initial_cash,
    log_y=True,
)
fig.show()

# Best pick parameter summary

- Show the selected metadata and weighting parameters clearly


In [ ]:
best_pick_parameters = pd.DataFrame(
    [{k: v for k, v in best_pick.combination.as_dict().items()}]
).T.rename(columns={0: "value"})
display(best_pick_parameters)


# Portfolio performance (best pick)

- Compare buy and hold against our best performance

In [ ]:
from tradeexecutor.analysis.multi_asset_benchmark import compare_strategy_backtest_to_multiple_assets

returns = best_pick.returns

df = compare_strategy_backtest_to_multiple_assets(
    state=state,
    strategy_universe=strategy_universe,
    returns=returns,
    display=True,
    asset_count=3,
)
display(df)

# Trade summary (best pick)

- Show statistics about the made trades

In [ ]:
summary = best_pick.summary
display(summary.to_dataframe())

# Trading pair performance breakdown

- Show breakdown of different pairs on the best result

In [ ]:
from tradeexecutor.analysis.multipair import analyse_multipair
from tradeexecutor.analysis.multipair import format_multipair_summary

multipair_summary = analyse_multipair(state, show_address=True)
display(format_multipair_summary(multipair_summary))

# PnL concentration

- Check how much of the result comes from the top vaults


In [ ]:
pnl_concentration = multipair_summary[["Trading pair", "Total PnL USD"]].copy()
pnl_concentration = pnl_concentration.sort_values("Total PnL USD", ascending=False)
total_pnl = pnl_concentration["Total PnL USD"].sum()
concentration_summary = pd.DataFrame([
    {
        "Total PnL USD": total_pnl,
        "Top 3 PnL share": pnl_concentration.head(3)["Total PnL USD"].sum() / total_pnl if total_pnl else np.nan,
        "Top 5 PnL share": pnl_concentration.head(5)["Total PnL USD"].sum() / total_pnl if total_pnl else np.nan,
    }
])
display(concentration_summary)
display(pnl_concentration.head(10))


# Rolling sharpe

- The rolling sharpe ratio of the best pick

In [ ]:
import plotly.express as px

from tradeexecutor.visual.equity_curve import calculate_equity_curve, calculate_returns
from tradeexecutor.visual.equity_curve import calculate_rolling_sharpe

rolling_sharpe = calculate_rolling_sharpe(
    returns,
    freq="D",
    periods=90,
)

fig = px.line(rolling_sharpe, title='Strategy rolling Sharpe (6 months)')
fig.update_layout(showlegend=False)
fig.update_yaxes(title="Sharpe")
fig.update_xaxes(title="Time")
fig.show()

# Backtesting chart rendering for the best strategy

- Charts from the best grid search result using chart renderer

In [ ]:
from tradeexecutor.strategy.chart.definition import ChartRegistry, ChartKind, ChartInput
from tradeexecutor.strategy.chart.renderer import ChartBacktestRenderingSetup
from tradeexecutor.strategy.chart.standard.weight import equity_curve_by_asset
from tradeexecutor.strategy.chart.standard.weight import equity_curve_by_chain
from tradeexecutor.strategy.chart.standard.weight import weight_allocation_statistics
from tradeexecutor.strategy.chart.standard.weight import volatile_weights_by_percent
from tradeexecutor.strategy.chart.standard.weight import volatile_and_non_volatile_percent
from tradeexecutor.strategy.chart.standard.profit_breakdown import trading_pair_breakdown
from tradeexecutor.strategy.chart.standard.interest import vault_statistics
from tradeexecutor.strategy.chart.standard.vault import all_vault_positions
from tradeexecutor.strategy.chart.standard.vault import all_vault_daily_gains_losses
from tradeexecutor.strategy.chart.standard.vault import vault_position_timeline
from tradeexecutor.strategy.chart.standard.trading_universe import available_trading_pairs
from tradeexecutor.strategy.pandas_trader.indicator import load_indicators_inline
from tradeexecutor.strategy.parameters import StrategyParameters
from plotly.graph_objects import Figure

# Merge full strategy parameters with best combination's searched values
# (same pattern the optimiser uses internally — see optimiser.py ObjectiveWrapper)
merged_parameters = StrategyParameters.from_class(Parameters)
merged_parameters.update(best_pick.combination.as_dict())

# Load cached indicators for the best strategy from disk
# (calculated during grid search, no recalculation needed)
indicator_data = load_indicators_inline(
    strategy_universe=strategy_universe,
    create_indicators=indicators.create_indicators,
    parameters=merged_parameters,
)


def trading_pair_breakdown_with_chain(input: ChartInput) -> pd.DataFrame:
    """Trading pair breakdown with chain column and address."""
    return trading_pair_breakdown(
        input,
        show_chain=True,
        show_address=True,
    )

def all_vault_positions_by_profit(input: ChartInput) -> pd.DataFrame:
    """Top 10 winning and bottom 10 losing vault positions."""
    return all_vault_positions(
        input,
        sort_by="Profit USD",
        sort_ascending=False,
        show_address=True,
        top_n=10,
        bottom_n=10,
    )

def biggest_daily_gains_losses(input: ChartInput) -> pd.DataFrame:
    """Top 10 biggest daily gains and bottom 10 biggest daily losses."""
    return all_vault_daily_gains_losses(input, top_n=10, bottom_n=10)


charts = ChartRegistry(default_benchmark_pairs=BENCHMARK_PAIRS)
charts.register(equity_curve_by_asset, ChartKind.state_all_pairs)
charts.register(equity_curve_by_chain, ChartKind.state_all_pairs)
charts.register(weight_allocation_statistics, ChartKind.state_all_pairs)
charts.register(volatile_weights_by_percent, ChartKind.state_all_pairs)
charts.register(volatile_and_non_volatile_percent, ChartKind.state_all_pairs)
charts.register(trading_pair_breakdown_with_chain, ChartKind.state_all_pairs)
charts.register(vault_statistics, ChartKind.state_all_pairs)
charts.register(all_vault_positions_by_profit, ChartKind.state_single_vault_pair)
charts.register(biggest_daily_gains_losses, ChartKind.state_single_vault_pair)
charts.register(vault_position_timeline, ChartKind.state_single_vault_pair)

charts.register(available_trading_pairs, ChartKind.indicator_all_pairs)
chart_renderer = ChartBacktestRenderingSetup(
    registry=charts,
    state=state,
    backtest_start_at=backtest_start,
    backtest_end_at=backtest_end,
    strategy_input_indicators=indicator_data,
)

# Rank diagnostics

- Break down the best pick into its component feature ranks and final composite score
- Measure the spread and stability of the rank-composite signal


In [ ]:
import plotly.express as px

feature_sets = {
    "returns + sharpe + age + tvl": ["rolling_returns", "rolling_sharpe", "age", "tvl"],
    "returns + sharpe + drawdown + age + tvl": ["rolling_returns", "rolling_sharpe", "drawdown_from_peak", "age", "tvl"],
    "returns + sterling + drawdown + age + tvl": ["rolling_returns", "rolling_sterling_floor", "drawdown_from_peak", "age", "tvl"],
}
feature_directions = {
    "rolling_returns": False,
    "rolling_sharpe": False,
    "rolling_sterling_floor": False,
    "age": False,
    "tvl": False,
    "drawdown_from_peak": False,
    "drawdown_duration": True,
}
selected_features = feature_sets[merged_parameters.rank_feature_set]
rank_weights = _build_rank_weights(merged_parameters.rank_feature_set, merged_parameters.rank_weight_scheme)

included_pair_series = indicator_data.get_indicator_series("inclusion_criteria")
latest_included_pair_ids = list(included_pair_series.dropna().iloc[-1]) if len(included_pair_series.dropna()) else []

rows = []
for pair_id in latest_included_pair_ids:
    pair = strategy_universe.get_pair_by_id(pair_id)
    gate_signal_series = indicator_data.get_indicator_series("rolling_calmar_transformed", pair=pair)
    gate_signal = gate_signal_series.dropna().iloc[-1] if gate_signal_series is not None and len(gate_signal_series.dropna()) else np.nan
    if pd.isna(gate_signal) or gate_signal <= 0:
        continue
    row = {
        "Vault": pair.base.token_symbol if hasattr(pair.base, "token_symbol") else pair.get_ticker(),
        "Pair id": pair_id,
        "Gate signal": gate_signal,
        "Age years": indicator_data.get_indicator_series("age", pair=pair).dropna().iloc[-1],
        "TVL": indicator_data.get_indicator_series("tvl", pair=pair).dropna().iloc[-1],
    }
    valid = True
    for feature in selected_features:
        value = indicator_data.get_indicator_series(feature, pair=pair)
        if value is None or len(value.dropna()) == 0:
            valid = False
            break
        row[feature] = value.dropna().iloc[-1]
    if valid:
        rows.append(row)

rank_df = pd.DataFrame(rows)
for feature in selected_features:
    rank_df[f"rank_{feature}"] = _rank_percentile(rank_df[feature], ascending=feature_directions[feature])
rank_df["rank_composite_signal"] = 0.0
for feature, weight in rank_weights.items():
    rank_df["rank_composite_signal"] += rank_df[f"rank_{feature}"] * weight

rank_df = rank_df.sort_values("rank_composite_signal", ascending=False)
display(rank_df.head(20))

rank_signal_history = []
for pair_id in latest_included_pair_ids:
    pair = strategy_universe.get_pair_by_id(pair_id)
    pair_data = {}
    valid = True
    for feature in selected_features:
        series = indicator_data.get_indicator_series(feature, pair=pair)
        if series is None:
            valid = False
            break
        pair_data[feature] = series
    if not valid:
        continue

feature_panels = []
for feature in selected_features:
    combined = indicator_data.get_indicator_data_pairs_combined(feature).unstack(level="pair_id")
    feature_panels.append(_rank_percentile(combined.rank(axis=1, method="average", ascending=feature_directions[feature], pct=True), ascending=False))

# Build direct spread diagnostics from the final rebalance cross-section and raw rank-composite history
rank_composite_current = rank_df.set_index("Pair id")["rank_composite_signal"]
display(rank_df[["Vault", "rank_composite_signal"] + [f"rank_{feature}" for feature in selected_features]].head(20))

rank_only = rank_df[[f"rank_{feature}" for feature in selected_features] + ["rank_composite_signal"]]
display(rank_only.corr())

rank_df["Age decile"] = pd.qcut(rank_df["Age years"].rank(method="first"), q=min(5, rank_df["Age years"].notna().sum()), duplicates="drop")
rank_df["TVL decile"] = pd.qcut(rank_df["TVL"].rank(method="first"), q=min(5, rank_df["TVL"].notna().sum()), duplicates="drop")
display(rank_df.groupby("Age decile", observed=False)[["rank_composite_signal"]].agg(["count", "mean"]))
display(rank_df.groupby("TVL decile", observed=False)[["rank_composite_signal"]].agg(["count", "mean"]))


# Available pairs (for the best strategy)

- Number of pairs available to trade every month

In [ ]:
fig, df = chart_renderer.render(available_trading_pairs, with_dataframe=True)
fig.show()
display(df.tail(5))

# Trading pair breakdown

- Trade success for each trading pair

In [ ]:
df = chart_renderer.render(trading_pair_breakdown_with_chain)
display(df)

# Vault performance

- Analyse the performance of our vaults

## Vault statistics

- Calculate interest accrued on different vaults for the strategy

In [ ]:
df = chart_renderer.render(vault_statistics)
display(df)

# Rolling Sharpe

- The rolling Sharpe ratio of the best strategy

In [ ]:
import plotly.express as px

from tradeexecutor.visual.equity_curve import calculate_equity_curve, calculate_returns
from tradeexecutor.visual.equity_curve import calculate_rolling_sharpe

equity_curve = calculate_equity_curve(state)
returns = calculate_returns(equity_curve)

rolling_sharpe = calculate_rolling_sharpe(
    returns,
    freq="D",
    periods=180,
)

fig = px.line(rolling_sharpe, title='Strategy rolling Sharpe (6 months)')
fig.update_layout(showlegend=False)
fig.update_yaxes(title="Sharpe")
fig.update_xaxes(title="Time")
fig.show()

# Asset weights

- What assets were allocated over time
- Do both proportional % and USD weights

## Portfolio equity curve breakdown by asset

- Where did we make the profit

In [ ]:
fig = chart_renderer.render(equity_curve_by_asset)
fig.show()

## Weight allocation statistics

In [ ]:
from tradeexecutor.utils.notebook import display_dataframe_with_html

stats = chart_renderer.render(weight_allocation_statistics)
display_dataframe_with_html(stats)

## Vault position list

- Biggest winning and losing positions taken in the vaults
- Biggest daily gains and losses by absolute USD jump

In [ ]:
df = chart_renderer.render(all_vault_positions_by_profit)
display(df)


In [ ]:
from tradeexecutor.utils.notebook import display_dataframe_with_html

df = chart_renderer.render(biggest_daily_gains_losses)
display_dataframe_with_html(df)
